In [ ]:
!pip install catboost -q


In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier


In [ ]:




# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(path + '/Q3_data.csv')


In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()


In [ ]:
# Task 4: Write your code here:
df.describe()


In [ ]:
# Task 1: Write your code here:
print("Missing Values:")
print(df.isnull().sum())

# Fill missing values with mean for numerical columns
df = df.fillna(df.mean(numeric_only=True))


In [ ]:
# Task 2: Write your code here:
print("Checking for duplicate rows...")
duplicate_rows = df.duplicated().sum()

if duplicate_rows > 0:
    print(f"Found {duplicate_rows} duplicate rows. Removing them...")
    df.drop_duplicates(inplace=True)
    print("Duplicate rows removed.")
else:
    print("No duplicate rows found.")


In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

# Check for categorical columns
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Categorical columns: {list(categorical_cols)}")

# Encode if any exist
for col in categorical_cols:
    if col != 'target':  # Don't encode the target variable
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])


In [ ]:
# Task 4: Write your code here:

X = df.drop('Target', axis=1)
y = df['Target']

scaler = StandardScaler() # Standardize features
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)


In [ ]:
# Task 5: Write your code here:
print("Target distribution:")
print(y.value_counts())
print("\nPercentages:")
print(y.value_counts(normalize=True))
if y.value_counts(normalize=True).min() < 0.3:
    print("\nThe dataset is IMBALANCED")
else:
    print("\nThe dataset is BALANCED")


In [ ]:
# Task 1: Write your code here:
X = X_scaled
y = df['Target']


In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
import numpy as np

# use stratified for imbalanced data
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for fold, (train_idx, val_idx) in enumerate(kfold.split(X, y)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # train model
    clf = CatBoostClassifier(iterations=100, verbose=0, random_state=42)
    clf.fit(X_train, y_train)

    # predict
    preds = clf.predict(X_val)

    # evaluate
    score = f1_score(y_val, preds)
    scores.append(score)
    print(f"Fold {fold+1} F1 Score: {score:.4f}")

print(f"\nAverage F1 Score: {np.mean(scores):.4f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
feature_importance = model.get_feature_importance()
features = X.columns
importance_df = pd.DataFrame({
    'feature': features,
    'importance': feature_importance
}).sort_values('importance', ascending=False)
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'][:10], importance_df['importance'][:10])
plt.xlabel('Importance')
plt.title('Top 10 Feature Importance')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
golden_feature = importance_df.iloc[0]['feature']
print(f"The Golden Feature is: {golden_feature}")

In [ ]:
# Task Bonus: Write your code here:
X_golden = X[[golden_feature]]
scores_golden = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_golden, y)):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    model_golden = CatBoostClassifier(iterations=100, verbose=0, random_state=42)
    model_golden.fit(X_train, y_train)
    y_pred = model_golden.predict(X_val)
    score = f1_score(y_val, y_pred)
    scores_golden.append(score)

print(f"Full Model Average F1: {np.mean(scores):.4f}")
print(f"Golden Feature Only F1: {np.mean(scores_golden):.4f}")
print(f"Difference: {np.mean(scores) - np.mean(scores_golden):.4f}")